# 2. Train / val / test pairs (cases 0–3)

Turns the ZIP-cleaned Overture ↔ Foursquare positives (`ct_positives_zip_cleaned`, from `01_overture_fsq.ipynb`) into labelled train / val / test address pairs, pandas only. Each stage reads the previous stage's parquet from `Data/`:

| § | Stage | Output (`Data/…_2026-09-23.1.parquet`) |
|---|---|---|
| 4 | Assign whole ZIPs to train/val/test | `ct_positives_split_data` (ZIP → split mapping) |
| 5 | Attach split, set aside `number_mismatch` rows | `ct_positives_remove_number_mismatch` (positives), `ct_positives_mismatch_holdout` |
| 6 | Hard negatives, Case 1: same street, different number | `ct_neg_case1` |
| 7 | Hard negatives, Case 2: same number, similar street, far apart | `ct_neg_case2` |
| 8a | Dedupe positives; easy negatives (Case 3) | `ct_positives_final`, `ct_neg_case3` |
| 8b | Final assembly | **`ct_pairs_train` / `ct_pairs_val` / `ct_pairs_test`** |

(§1–3 are the last stages of `01_overture_fsq.ipynb`; the numbering is kept.) Negatives always pair `address_a_clean` of place X with `address_b_clean` of place Y (A = Overture, B = Foursquare), so cross-source formatting appears in both classes. All sampling uses `random_state=42`. Run top to bottom: §7's cells modify `pairs` in place, so rerun them starting from its first cell.

The paper's dataset adds look-alike street swaps to these files: `03_case2s.ipynb`.

## 0. Setup

In [ ]:
import pandas as pd

OVERTURE_RELEASE = "2026-09-23.1"   # must match 01_overture_fsq.ipynb

## 4. Train / val / test split by ZIP

Whole ZIPs go to one split, assigned **before** any pairs are generated:
- no address string can appear in two splits (no leakage);
- neighbours on the same street stay in the same split, so same-street hard negatives can still be built;
- the test set is neighbourhoods the model has never seen.

ZIPs are shuffled (seeded), then cut by **cumulative row share** (not ZIP count, since ZIP sizes range 1–763 rows).
The largest ZIP holds only ~2% of rows, so no single ZIP can swamp a split.

Result: train / val / test = **70.0% / 14.8% / 15.2%** of rows, **228 / 47 / 56** ZIPs.

In [ ]:
current_zip = pd.read_parquet(f'Data/ct_positives_zip_cleaned_{OVERTURE_RELEASE}.parquet')

# One row per ZIP with its row count, in random (seeded) order
zip_freq = current_zip.groupby('zip_5').size().sample(frac=1, random_state=42).reset_index(name= 'n')

# Walk down the shuffled list: running share of rows decides the split (0.70 / 0.85 cut points)
zip_freq["running_frac"] = zip_freq["n"].cumsum() / zip_freq["n"].sum()
zip_freq["split"] = pd.cut(zip_freq["running_frac"], bins=[0,0.7,0.85,1.0], labels = ["train", "val", "test"])
zip_freq.to_parquet(f'Data/ct_positives_split_data_{OVERTURE_RELEASE}.parquet')

## 5. Attach the split; set aside `number_mismatch` rows

Every place gets its ZIP's split (checked: 37,075 rows kept, 0 missing, 25,950 / 5,483 / 5,642).

`number_mismatch` rows (~5% of each split) are excluded from **all** splits: their labels are unreliable, and at
this size they'd move match recall by more than the differences between models. They are saved separately for the
hand-labelled test subset.
- positives: **35,298** (train 24,673 / val 5,245 / test 5,380) — `label` is already 1
- holdout: **1,777** (train 1,277 / val 238 / test 262)

In [ ]:
current_zip = pd.read_parquet(f'Data/ct_positives_zip_cleaned_{OVERTURE_RELEASE}.parquet')
zip_freq = pd.read_parquet(f'Data/ct_positives_split_data_{OVERTURE_RELEASE}.parquet')
new_zip = current_zip.merge(zip_freq[['zip_5', 'split']], left_on='zip_5', right_on='zip_5')
new_zip.head()

In [ ]:
# Positives used to build the dataset
no_number_mismatch = new_zip[new_zip['number_mismatch'] == False]
no_number_mismatch.to_parquet(f'Data/ct_positives_remove_number_mismatch_{OVERTURE_RELEASE}.parquet')

# Set aside for hand-labelling
yes_number_mismatch = new_zip[new_zip['number_mismatch'] == True]
yes_number_mismatch.to_parquet(f'Data/ct_positives_mismatch_holdout_{OVERTURE_RELEASE}.parquet')

## 6. Hard negatives — Case 1: same street, different number

e.g. `12 ELM ST` vs `14 ELM ST`. Self-join places on **(split, `zip_5`, `street_a`)** — same street *within one ZIP*,
so `MAIN ST` in two different towns never pairs — then keep pairs with different places and different numbers.

- Uncapped candidates (n·(n−1) per street): 812,888 / 199,234 / 161,932 — ~3.7× the whole negative budget,
  dominated by commercial corridors (`POST RD E` alone: 225 places).
- Text-pair dedupe **before** the cap: several businesses in one building produce identical address strings
  (about a quarter of capped pairs at k=4 were duplicates).
- **Per-place cap k=5** (each place X gets ≤5 random partners), so big streets grow linearly, not quadratically.

Result: **95,432** (train 67,195 / val 13,781 / test 14,456) ≈ 30% of each split's 9× negative budget.

In [ ]:
df = pd.read_parquet(f'Data/ct_positives_remove_number_mismatch_{OVERTURE_RELEASE}.parquet')

# Split side A into house number and street: "12 ELM ST" -> "12", "ELM ST"
# (no-number addresses: number_a = NaN, street_a = whole string)
df["number_a"] = df["address_a_clean"].str.extract(r"^(\d+)", expand = False)
df["street_a"] = df["address_a_clean"].str.replace(r"^\d+\s+", "", regex = True)

# Exploration: street-group sizes and uncapped pair counts per split (used to choose the cap)
# s = df.groupby(["split", "zip_5", "street_a"], observed = True).size().sort_values(ascending = False)
# print(len(s[s >= 2]))
# print((s * (s-1)).groupby(level = "split", observed = True).sum())

# Self-join: every pair of places on the same street, in the same ZIP and split
pairs = df.merge(df, on = ["split", "zip_5", "street_a"], suffixes = ("_x", "_y"))

pairs = pairs[
    (pairs["gers_id_x"] != pairs["gers_id_y"]) &                    # different places
    pairs["number_a_x"].notna() &                                   # both have a number
    pairs["number_a_y"].notna() &                                   #   (NaN != x is True in pandas)
    (pairs["number_a_x"] != pairs["number_a_y"]) &                  # different numbers
    (pairs["address_a_clean_x"] != pairs["address_b_clean_y"])     # not the same building (building-level match)
]

# Dedupe identical text pairs, then keep at most 5 random partners per place X
pairs = pairs.drop_duplicates(["address_a_clean_x", "address_b_clean_y"]).sample(frac = 1, random_state = 42).groupby("gers_id_x").head(5)

# Negative row = A of place X vs B of place Y
ct_neg_case1 = pairs[["address_a_clean_x", "address_b_clean_y", "zip_5", "split", "gers_id_x", "gers_id_y"]].rename(
    columns = {
        "address_a_clean_x": "address_a",
        "address_b_clean_y": "address_b"
    }
)
ct_neg_case1["label"] = 0
print(ct_neg_case1.groupby("split", observed = True).size())
print(len(ct_neg_case1[ct_neg_case1["label"] != 0]))   # should be 0
ct_neg_case1.to_parquet(f"Data/ct_neg_case1_{OVERTURE_RELEASE}.parquet")

## 7. Hard negatives — Case 2: same number, similar street, far apart

e.g. `100 N MAIN ST` vs `100 S MAIN ST`. Self-join on **(split, `zip_5`, `number_a`)**, keep different streets
(43,418 candidates: 32,198 / 5,778 / 5,442), score street similarity with Jaro-Winkler.

**String similarity alone is not enough on real data.** The highest-scoring pairs were the *same* street spelled two
ways (`KILLINGWORTH TURNPIKE` / `TURNPIK`, `20A` / `20-A`, `CUTOFF` / `CUT-OFF`) — matches, not negatives.
Distance between the two places (from Overture `bbox`, haversine) separates the two situations:
~99% of pairs with JW > 0.85 are within 500 m — different businesses at the same property (mall / plaza pins sit
50–200 m apart). Only the far pairs are genuinely different streets; a seeded sample was mostly directional variants
(`N MAIN ST` / `S MAIN ST`, `MAIN ST` / `W MAIN ST`).

Rule: **JW > 0.80 and distance > 500 m**, then the same exclusion / dedupe / k=5 cap as Case 1.
Result: **242** (train 180 / val 39 / test 23).

In [ ]:
df = pd.read_parquet(f'Data/ct_positives_remove_number_mismatch_{OVERTURE_RELEASE}.parquet')
df["number_a"] = df["address_a_clean"].str.extract(r"^(\d+)", expand = False)
df["street_a"] = df["address_a_clean"].str.replace(r"^\d+\s+", "", regex = True)
df = df[df["number_a"].notna()]   # before the merge: merge treats NaN keys as equal
print(df.columns)

# Self-join: every pair of places with the same number, in the same ZIP and split
pairs = df.merge(df, on = ["split", "zip_5", "number_a"], suffixes = ("_x", "_y"))

pairs = pairs[
    (pairs["gers_id_x"] != pairs["gers_id_y"]) &      # different places
    (pairs["street_a_x"] != pairs["street_a_y"])      # different street (same street + same number = same building)
]

print(len(pairs))
print(pairs.groupby("split", observed = True).size())

In [ ]:
# NOTE: modifies `pairs` in place — rerun the previous cell first, or the coordinate columns get merged twice.
import jellyfish

# Street similarity (streets only: the shared house number would inflate Jaro-Winkler's prefix bonus)
pairs["jw_pct"] = pairs.apply(
    lambda row:
    jellyfish.jaro_winkler_similarity(row["street_a_x"], row["street_a_y"]),
    axis = 1
)

# Coordinates of each place: Overture bbox of a point place (x = lon, y = lat)
coords = pd.read_parquet(f'Data/ct_places_{OVERTURE_RELEASE}.parquet')[["id", "bbox"]]

# Attach to X, then Y (2nd merge's overlapping id/bbox columns become id_x/bbox_x and id_y/bbox_y)
pairs = pairs.merge(coords, left_on = "gers_id_x", right_on = "id")
pairs = pairs.merge(coords, left_on = "gers_id_y", right_on = "id", suffixes = ("_x", "_y"))

In [ ]:
import numpy as np

pairs["lat_x"] = pairs["bbox_x"].str["ymin"]
pairs["lon_x"] = pairs["bbox_x"].str["xmin"]
pairs["lat_y"] = pairs["bbox_y"].str["ymin"]
pairs["lon_y"] = pairs["bbox_y"].str["xmin"]

# Haversine distance in meters (trig needs radians)
pairs["lat_delta"] = pairs["lat_x"] - pairs["lat_y"]
pairs["lon_delta"] = pairs["lon_x"] - pairs["lon_y"]
pairs["haver_a"] = (
    np.sin(np.radians(pairs["lat_delta"])/2)**2 +
    np.cos(np.radians(pairs["lat_x"])) * np.cos(np.radians(pairs["lat_y"])) *
    np.sin(np.radians(pairs["lon_delta"])/2)**2)
R = 6371000   # Earth radius, m
pairs["dist_m"] = 2 * R * np.arcsin(np.sqrt(pairs["haver_a"]))

In [ ]:
ct_neg_case2 = (pairs[(pairs["jw_pct"] > 0.8) &                             # similar street names
    (pairs["dist_m"] > 500) &                                      # but genuinely different places
    (pairs["address_a_clean_x"] != pairs["address_b_clean_y"])]    # exclusion rule
    .drop_duplicates(subset = ["address_a_clean_x", "address_b_clean_y"])
    .sample(frac=1, random_state = 42)
    .groupby("gers_id_x")
    .head(5)                                                       # same k=5 cap as Case 1
[["address_a_clean_x", "address_b_clean_y", "zip_5", "split", "gers_id_x", "gers_id_y"]]
.rename(columns = {"address_a_clean_x": "address_a", "address_b_clean_y": "address_b"})
                    )
ct_neg_case2["label"] = 0
# ct_neg_case2.groupby("split", observed = True).size()

ct_neg_case2.to_parquet(f'Data/ct_neg_case2_{OVERTURE_RELEASE}.parquet')

## 8. Easy negatives + final assembly

### 8a. Dedupe positives, then easy negatives (Case 3)

**Positive dedupe:** 23% of positives were exact repeats of the same text pair — many businesses at one big address
(`7 BACKUS AVE | 7 BACKUS AVE` × 77, `1 MOHEGAN SUN BLVD` × 40). To the model those are one example, so positives are
deduped **within each split** on what the model sees (both addresses + both ZIPs): **35,298 → 28,834**
(train 20,261 / val 4,187 / test 4,386). Done *before* the easy budget so the 1:9 ratio uses deduped counts.

**Easy negatives:** per split, budget = 9 × positives − Case 1 − Case 2. X and Y drawn at random **with replacement
from the same split** (separate seeds), placed side by side, then: different place, exclusion rule
(A of X ≠ B of Y), not already a positive / Case 1 / Case 2 text pair, dedupe, trim to the exact budget
(1.2× over-draw was enough). Result **163,832** (114,974 / 23,863 / 24,995); ~2% of easy pairs share a ZIP.

In [ ]:
# Dedupe positives within each split on the model's inputs (addresses + ZIPs); saved for assembly
positives_before = pd.read_parquet(f'Data/ct_positives_remove_number_mismatch_{OVERTURE_RELEASE}.parquet')
print(len(positives_before))
positives = positives_before.drop_duplicates(subset=["split", "address_a_clean", "address_b_clean", "zip_5", "zip_b"])
print(len(positives))
positives.to_parquet(f'Data/ct_positives_final_{OVERTURE_RELEASE}.parquet')


case1 = pd.read_parquet(f'Data/ct_neg_case1_{OVERTURE_RELEASE}.parquet')
case2 = pd.read_parquet(f'Data/ct_neg_case2_{OVERTURE_RELEASE}.parquet')


# Text-pair keys that easy negatives must not repeat: positives, Case 1, Case 2 (A|B, clean values)
check0 = positives.apply(lambda x: x["address_a_clean"] + "|" + x["address_b_clean"], axis = 1)
check1 = case1.apply(lambda x: x["address_a"] + "|"+ x["address_b"], axis = 1)
check2 = case2.apply(lambda x: x["address_a"] + "|" + x["address_b"], axis = 1)

check_all = pd.concat([check0, check1, check2])

# Easy budget per split = 9 x positives - Case 1 - Case 2
total_remaining = 9*positives.groupby("split", observed = True).size() - case1.groupby("split", observed = True).size() - case2.groupby("split", observed = True).size()
ret = pd.DataFrame()
for split, freq in total_remaining.items():
    left = (positives[positives["split"] == split]
            .sample(int(1.2*freq), replace= True, random_state = 42)
            .reset_index(drop = True)
            .add_suffix("_x")
        )
    right = (positives[positives["split"] == split]
             .sample(int(1.2*freq),replace= True, random_state = 24)
             .reset_index(drop = True)
             .add_suffix("_y")
             )
    # X and Y: random places from this split only, side by side (row i of left vs row i of right)
    merged = pd.concat([left, right], axis = 1)
    merged["combined_address"] = merged.apply(lambda x: x["address_a_clean_x"] + "|" + x["address_b_clean_y"], axis = 1)
    merged = (
        merged 
        [
            (merged["address_a_clean_x"] != merged["address_b_clean_y"]) &     # exclusion rule
            (merged["gers_id_x"] != merged["gers_id_y"]) &                     # different places
            (~merged["combined_address"].isin(check_all))                         # not a positive / Case 1 / Case 2 pair
        ]
         .sample(frac = 1, random_state = 34)
    .drop_duplicates(subset = ["combined_address"])
    .head(freq)
    )
    ret = pd.concat([ret, merged], axis = 0)

ret = ret[["address_a_clean_x", "address_b_clean_y", "zip_5_x", "split_x", "gers_id_x", "gers_id_y"]].rename(
    columns = {
        "address_a_clean_x":"address_a", 
        "address_b_clean_y": "address_b",
        "zip_5_x": "zip_5",
        "split_x": "split"    
    }
)
ret["label"] = 0
print(ret.groupby("split", observed = True).size())

ret.to_parquet(f'Data/ct_neg_case3_{OVERTURE_RELEASE}.parquet')

### 8b. Final assembly

- Positives (`neg_type = case0`) from the **deduped** file; Case 1 / 2 / 3 negatives from their files.
- Every row gets `zip_a` (X's Overture ZIP = the old `zip_5`) and `zip_b` — for negatives, **Y's Foursquare ZIP**,
  looked up by `gers_id_y` in the **full** (un-deduped) positives table: some Ys were removed as duplicate positives
  (10 Case 2 rows would otherwise be silently dropped by the inner merge).
- Negatives whose `address_a|address_b` text equals a positive's are removed (contradictory labels): **46**, all from
  Case 1 / 2 (a third place Z happens to have exactly X's A and Y's B).

In [ ]:
positives = pd.read_parquet(f'Data/ct_positives_final_{OVERTURE_RELEASE}.parquet')   # deduped -> case0
# Y's Foursquare ZIP, from the FULL positives table (some Ys were deduped away)
merge_pos_zip_b = pd.read_parquet(f'Data/ct_positives_remove_number_mismatch_{OVERTURE_RELEASE}.parquet')[["gers_id", "zip_b"]]

case1 = (pd.read_parquet(f'Data/ct_neg_case1_{OVERTURE_RELEASE}.parquet')
         .rename(columns = {"zip_5":"zip_a"})
         )
case1 = (case1.merge(merge_pos_zip_b, left_on = "gers_id_y", right_on = "gers_id")
         .drop(columns = "gers_id")
         )
case2 = (pd.read_parquet(f'Data/ct_neg_case2_{OVERTURE_RELEASE}.parquet')
         .rename(columns = {"zip_5":"zip_a"})
        )
case2 = (case2.merge(merge_pos_zip_b, left_on = "gers_id_y", right_on = "gers_id")
         .drop(columns = "gers_id")
         )
case3 = (pd.read_parquet(f'Data/ct_neg_case3_{OVERTURE_RELEASE}.parquet')
         .rename(columns = {"zip_5":"zip_a"})
         )
case3 = (case3.merge(merge_pos_zip_b, left_on = "gers_id_y", right_on = "gers_id")
        .drop(columns = "gers_id")
         )

# Positives in the same shape as the negatives (one place, so gers_id_x = gers_id_y)
case0 = (
    positives[["address_a_clean", "address_b_clean", "zip_5", "zip_b", "split"]]
    .rename(columns = {
        "address_a_clean": "address_a",
        "address_b_clean": "address_b",
        "zip_5" : "zip_a"
    })
         )
case0["gers_id_x"] = positives["gers_id"]
case0["gers_id_y"] = positives["gers_id"]
case0["label"] = 1

neg_types = [
    ["case0", case0],
    ["case1", case1],
    ["case2", case2],
    ["case3", case3]
] 
for neg_type, neg_df in neg_types:
    neg_df["neg_type"] = neg_type

final_dataset = pd.concat([case0, case1, case2, case3], ignore_index= True)
final_dataset["combined_address"] = final_dataset.apply(lambda x: x["address_a"]+"|"+x["address_b"], axis=1)
# Drop negatives whose text pair is also a positive
pos_keys = final_dataset[final_dataset["label"] == 1]["combined_address"]
conflict = (final_dataset["label"] == 0) & (final_dataset["combined_address"].isin(pos_keys))
print(conflict.sum())

final_dataset = final_dataset[~conflict]
print(len(final_dataset[          # negatives with identical A and B: should be 0
    (final_dataset["address_a"] == final_dataset["address_b"]) & 
    (final_dataset["label"] == 0)
]))
print(pd.crosstab(final_dataset["split"], final_dataset["neg_type"]))
print(final_dataset.duplicated(    # duplicate model inputs: should be 0
    ["address_a", "address_b", "zip_a", "zip_b"]).sum())
final_dataset = final_dataset.drop(columns = "combined_address")
final_dataset = final_dataset[~final_dataset["zip_b"].isna()]
# Drop pairs with an empty / whitespace-only address on either side: the siamese encoder
# can't take a length-0 string (pack_padded_sequence). 0 such rows when added.
empty = (final_dataset["address_a"].fillna("").str.strip() == "") | (final_dataset["address_b"].fillna("").str.strip() == "")
print("dropped (empty address):", empty.sum())
final_dataset = final_dataset[~empty]
# Shared pair schema (helpers/pairs.py): id_a/id_b, not the merge's _x/_y
final_dataset = final_dataset.rename(columns = {"gers_id_x": "id_a", "gers_id_y": "id_b"})[
    ["id_a", "id_b", "address_a", "address_b", "zip_a", "zip_b", "split", "label", "neg_type"]]
for split in ["train", "val", "test"]:
    final_dataset[final_dataset["split"] == split].to_parquet(f'Data/ct_pairs_{split}_{OVERTURE_RELEASE}.parquet')

### Result

| split | positives | Case 1 | Case 2 | easy | neg : pos |
|---|---|---|---|---|---|
| train | 20,261 | 67,178 | 172 | 114,974 | 9.00 |
| val | 4,187 | 13,781 | 34 | 23,863 | 9.00 |
| test | 4,386 | 14,448 | 15 | 24,995 | 9.00 |

288,294 rows. Verified on the saved files: 0 duplicate (addresses + ZIPs) rows, 0 positive/negative text conflicts,
0 negatives with `address_a == address_b`, `split` matches each file. Columns: `address_a`, `address_b`, `zip_a`,
`zip_b`, `split`, `gers_id_x`, `gers_id_y`, `label`, `neg_type`.

Notes: `zip_b` has 17 NULLs (features must handle missing). 1,603 rows share an address pair with another row but
differ in ZIP (different model inputs). A few hundred address strings occur in two splits — almost all the same string
in different towns (`71 MAIN ST`), but `500 W FARMS MALL` (Westfarms, on the Farmington / West Hartford line) is one
physical place in both train and test: a ZIP split can't fully separate places that straddle ZIP boundaries.